# NIH RePORTER MCP v0.14.6 — evidence, reconciliation, and optional agents

**Status: revised evaluation instrument; execution outputs have been cleared.**
**Instrument revision: wrap-result-fix-2026-10-21.** This fixes the notebook's
handling of FastMCP result objects, including union-return wrapping; it does not
change the server release.
This notebook targets tag **v0.14.6**, expects **14 tools, 9 prompts, and
3 resources**, and verifies source identity before importing it: the tag must
resolve to a commit whose **parent** matches the parent recorded in RUN_MANIFEST §1
at the tag (or an explicitly configured pin). A commit cannot contain its own SHA,
and commit SHAs appear only in RUN_MANIFEST, so the resolved SHA is recorded, not
hardcoded here.
The reported model-free CI baseline is **155 passed + 2 xfailed**. That is an
expected result, not a result produced by this revised notebook until you run it.

The core story is **define a cohort → show its evidence → independently
reconcile the funding → explain the limits**. Historical fixture replay,
fresh source capture, notebook-defined live checks, and model evaluation have distinct scopes.
Agent demonstrations and benchmarks are optional and require Databricks model
endpoints. They are not part of model-free CI.

This revision changes the notebook instrument, not the server at the release.
Its revised prompts, guardrails, and reporting protocol mean agent results are
not directly interchangeable with earlier notebook results.

Run from a fresh kernel, top to bottom. Core execution needs Python 3.11 or 3.12,
Git, package-index access, and a writable temporary directory. Live sections
also need upstream API access. No Databricks credentials are needed for core
checks. The optional section uses Databricks notebook credentials or explicit
environment credentials and is skipped by default.


## 1. Configuration and execution boundaries

For an offline demonstration after setup, disable RUN_LIVE_CAPTURE and
RUN_LIVE_SMOKE. Leave agent evaluations off unless the endpoints are available.
Every skipped section is recorded as **NOT_RUN**.

**Target: a new local HTTP MCP server from the pinned checkout.** An already
running Databricks App is neither discovered nor used. The notebook removes the
inherited DATABRICKS_APP_PORT and chooses a separate local port. Frozen replay
uses an in-process MCP server; notebook-defined live checks use the new local HTTP server.
The deployed app's Ask interface, service-principal authorization, and endpoint
health are not evaluated. RUN_AGENT_EVALS=False also leaves model calls untested.

Set ARTIFACT_BASE to an existing writable Unity Catalog Volume path for durable
Databricks outputs. Otherwise outputs remain in the run's temporary directory
and a ZIP is produced for download. Source and dependencies stay in driver-local
storage. Every execution uses a new directory.

Tagged requirements contain version ranges. CONSTRAINTS_FILE may point to an
independently verified constraints file. Without one, this is a **source-pinned
run with newly resolved dependencies**, not a fully locked reproduction.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import atexit
import hashlib
import json
import os
import platform
import signal
import shutil
import socket
import subprocess
import sys
import tempfile
import time
import uuid

REPOSITORY_URL = "https://github.com/conroyriodss/nih-reporter-mcp.git"
RELEASE_TAG = "v0.14.6"
RELEASE_VERSION = RELEASE_TAG.removeprefix("v")
# A commit cannot contain its own SHA, and commit SHAs appear only in RUN_MANIFEST.
# The tag must resolve to a commit whose parent equals the parent recorded in
# RUN_MANIFEST section 1 at the tag. Optional explicit pins (40-hex strings):
# EXPECTED_PARENT overrides the manifest value; EXPECTED_COMMIT pins the tag itself.
EXPECTED_PARENT = None
EXPECTED_COMMIT = None
EXPECTED_INVENTORY = {"tools": 14, "prompts": 9, "resources": 3}
EXPECTED_CI = {"passed": 155, "xfail": 2}

RUN_CI = True
RUN_LIVE_CAPTURE = True
RUN_LIVE_SMOKE = True
RUN_AGENT_EVALS = False
ARTIFACT_BASE = None  # Example: "/Volumes/catalog/schema/volume/nih-evaluations"
CONSTRAINTS_FILE = None  # Existing verified pip constraints file; optional.

DEMO_MODEL = "databricks-gpt-5-5"
EVAL_MODELS = [
    "databricks-gpt-5-5",
    "databricks-claude-sonnet-4-6",
    "databricks-meta-llama-3-3-70b-instruct",
]
MODEL_PARAMETERS = {}  # Supported settings actually sent; unset means provider default.

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
RUN_ROOT = Path(tempfile.mkdtemp(prefix="nih-" + RELEASE_TAG.replace(".", "") + "-"))
REPO_DIR = RUN_ROOT / "repo"
SITE_DIR = RUN_ROOT / "site"
ARTIFACT_DIR = (Path(ARTIFACT_BASE) / RUN_ID) if ARTIFACT_BASE else RUN_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=False)
print("Run ID:", RUN_ID)
print("Artifacts:", ARTIFACT_DIR)
print("Optional model evaluations:", RUN_AGENT_EVALS)
print("Evaluation target: new local pinned MCP server; deployed App/Ask is NOT tested.")
if not ARTIFACT_BASE:
    print("Outputs are temporary until you retain the final ZIP.")


## 2. Fetch and verify the release source

This uses a fresh checkout, not a copied personal workspace or cached Python
module. The tag and expected full commit must agree. Identity mismatch is a
stop condition; do not substitute another checkout under the same run ID.


In [ ]:
def checked_command(args, *, cwd=None, env=None, log_name=None):
    result = subprocess.run(args, cwd=cwd, env=env, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if log_name:
        (ARTIFACT_DIR / log_name).write_text(result.stdout, encoding="utf-8")
    if result.returncode:
        print(result.stdout[-6000:])
        raise RuntimeError(f"Command failed with exit status {result.returncode}: {args}")
    return result.stdout.strip()

if sys.version_info[:2] not in {(3, 11), (3, 12)}:
    raise RuntimeError("Use Python 3.11 or 3.12 to match the tagged CI matrix.")
if not shutil.which("git"):
    raise RuntimeError("Git is required in the execution environment.")

checked_command(["git", "clone", "--no-checkout", REPOSITORY_URL, str(REPO_DIR)],
                log_name="git-clone.log")
resolved = checked_command(["git", "rev-parse", RELEASE_TAG + "^{commit}"], cwd=REPO_DIR)
resolved_parent = checked_command(["git", "rev-parse", resolved + "^"], cwd=REPO_DIR)
if EXPECTED_PARENT:
    expected_parent, parent_source = EXPECTED_PARENT, "notebook configuration"
else:
    import re
    manifest_text = checked_command(
        ["git", "show", resolved + ":docs/demonstration_fixture/RUN_MANIFEST.md"], cwd=REPO_DIR)
    match = re.search(r"records the tagged commit's \*\*parent\*\*, `([0-9a-f]{40})`", manifest_text)
    if not match:
        raise RuntimeError("RUN_MANIFEST section 1 parent row not found; set EXPECTED_PARENT explicitly.")
    expected_parent, parent_source = match.group(1), "RUN_MANIFEST section 1 at the tag"
if resolved_parent != expected_parent:
    raise RuntimeError(f"{RELEASE_TAG} resolves to {resolved} with parent {resolved_parent}; "
                       f"expected parent {expected_parent} ({parent_source})")
if EXPECTED_COMMIT and resolved != EXPECTED_COMMIT:
    raise RuntimeError(f"Tag resolves to {resolved}, expected {EXPECTED_COMMIT}")
checked_command(["git", "checkout", "--detach", resolved], cwd=REPO_DIR)
if checked_command(["git", "status", "--porcelain"], cwd=REPO_DIR):
    raise RuntimeError("Release checkout is not clean.")
SOURCE_IDENTITY = {
    "tag": RELEASE_TAG, "evaluated_sha": resolved,
    "parent": checked_command(["git", "rev-parse", "HEAD^"], cwd=REPO_DIR),
    "parent_check_source": parent_source,
    "repository": REPOSITORY_URL,
    "tag_object": checked_command(["git", "rev-parse", RELEASE_TAG], cwd=REPO_DIR),
}
(ARTIFACT_DIR / "source_identity.json").write_text(json.dumps(SOURCE_IDENTITY, indent=2))
print(json.dumps(SOURCE_IDENTITY, indent=2))


## 3. Isolated dependencies and cold-import identity

Packages are installed into a fresh target directory. Server/evaluation
processes use Python's -S option and an explicit import path, excluding the
notebook runtime's site-packages. This avoids changing Databricks' preinstalled
MLflow dependencies and needs no notebook restart or virtual environment.

Additional pytest and JSON Schema packages belong to this evaluation instrument.
Their versions and all resolved runtime versions are recorded. Dependency
resolution failure is an environment failure, not a test pass.


In [ ]:
requirements = REPO_DIR / "requirements.txt"
if not requirements.exists():
    raise RuntimeError("Tagged requirements.txt is missing; verify the release installation instructions.")
install = [
    sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
    "--ignore-installed", "--target", str(SITE_DIR),
    "-r", str(requirements), "pytest==8.4.2", "jsonschema==4.25.1",
]
if CONSTRAINTS_FILE:
    constraints = Path(CONSTRAINTS_FILE).expanduser().resolve()
    if not constraints.is_file():
        raise RuntimeError("Configured constraints file does not exist.")
    install += ["-c", str(constraints)]
    shutil.copy2(constraints, ARTIFACT_DIR / "provided-constraints.txt")
checked_command(install, cwd=REPO_DIR, log_name="dependency-install.log")

RUNTIME_ENV = dict(os.environ)
RUNTIME_ENV["PYTHONPATH"] = os.pathsep.join([str(REPO_DIR / "src"), str(SITE_DIR)])
RUNTIME_ENV["PYTHONDONTWRITEBYTECODE"] = "1"
RUNTIME_ENV["PYTHONNOUSERSITE"] = "1"
RUNTIME_ENV["NIH_MCP_CACHE_DISABLED"] = "0"
RUNTIME_ENV.pop("DATABRICKS_APP_PORT", None)
RUNTIME_ENV.pop("PYTEST_ADDOPTS", None)

probe = '''
import importlib.metadata as m, json, sys
import nih_reporter_mcp, fastmcp
print(json.dumps({
  "python": sys.version, "executable": sys.executable,
  "server_module": nih_reporter_mcp.__file__,
  "server_version": nih_reporter_mcp.__version__,
  "fastmcp_module": fastmcp.__file__,
  "distributions": sorted(
      [{"name": d.metadata["Name"], "version": d.version}
       for d in m.distributions(path=[sys.argv[1]])], key=lambda d: d["name"].lower())
}))
'''
ENVIRONMENT = json.loads(checked_command(
    [sys.executable, "-S", "-c", probe, str(SITE_DIR)], env=RUNTIME_ENV))
if ENVIRONMENT["server_version"] != RELEASE_TAG.removeprefix("v"):
    raise RuntimeError("Imported server version does not match the release.")
if not Path(ENVIRONMENT["server_module"]).resolve().is_relative_to(REPO_DIR.resolve()):
    raise RuntimeError("Imported server came from outside the pinned checkout.")
ENVIRONMENT.update({
    "platform": platform.platform(),
    "requirements_sha256": hashlib.sha256(requirements.read_bytes()).hexdigest(),
    "dependency_resolution": "provided_constraints" if CONSTRAINTS_FILE else "fresh_resolution_from_tagged_ranges",
    "install_command": install,
})
(ARTIFACT_DIR / "environment.json").write_text(json.dumps(ENVIRONMENT, indent=2))
(ARTIFACT_DIR / "resolved-requirements.txt").write_text(
    "\n".join(f"{d['name']}=={d['version']}" for d in ENVIRONMENT["distributions"]) + "\n")
print("Imported:", ENVIRONMENT["server_module"])
print("Version:", ENVIRONMENT["server_version"])
print("Dependency resolution:", ENVIRONMENT["dependency_resolution"])
print("Distributions recorded:", len(ENVIRONMENT["distributions"]))


## 4. Notebook-owned evaluation implementation

The following collapsed cell writes the auditable worker: the independent
Decimal ledger, MCP replay adapter, live checks, argument validation, and
optional agent instrumentation. Expand it to inspect the code. Its hash is
recorded in the manifest. It does **not** edit tagged server source.

Each stage uses a fresh subprocess. Failures are saved and displayed; the final
gate raises after packaging evidence so failed runs remain inspectable.


In [ ]:
WORKER_SOURCE = '"""Notebook-owned evaluation instrument. It does not modify the tagged server."""\nfrom __future__ import annotations\n\nimport argparse\nimport asyncio\nimport copy\nimport csv\nimport dataclasses\nimport hashlib\nimport importlib.metadata\nimport json\nimport os\nimport re\nimport subprocess\nimport sys\nimport time\nimport traceback\nfrom collections import Counter\nfrom datetime import datetime, timezone\nfrom decimal import Decimal\nfrom pathlib import Path\nfrom unittest.mock import patch\n\nROOT = None\nCFG = None\nSTAGE = None\n\n\nclass ArtifactSerializationError(TypeError):\n    """The evaluation instrument could not preserve a returned value as JSON."""\n\n\nclass MCPToolResultError(RuntimeError):\n    """The MCP response explicitly reported a tool error."""\n\n\ndef utc():\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef plain(value):\n    if hasattr(value, "model_dump"):\n        return plain(value.model_dump(mode="json"))\n    if dataclasses.is_dataclass(value) and not isinstance(value, type):\n        return {f.name: plain(getattr(value, f.name)) for f in dataclasses.fields(value)}\n    if isinstance(value, dict):\n        return {str(k): plain(v) for k, v in value.items()}\n    if isinstance(value, (list, tuple, set)):\n        return [plain(v) for v in value]\n    if isinstance(value, Decimal):\n        return str(value)\n    if isinstance(value, Path):\n        return str(value)\n    if value is None or isinstance(value, (str, int, float, bool)):\n        return value\n    raise ArtifactSerializationError(f"Unsupported artifact value: {type(value).__name__}")\n\n\ndef save(name, value):\n    path = ROOT / name\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(plain(value), indent=2, ensure_ascii=False), encoding="utf-8")\n    return str(path.relative_to(ROOT))\n\n\ndef append_event(name, value):\n    path = ROOT / name\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("a", encoding="utf-8") as f:\n        f.write(json.dumps(plain(value), ensure_ascii=False) + "\\n")\n\n\ndef require(condition, message):\n    if not condition:\n        raise AssertionError(message)\n\n\ndef sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef field(obj, *names):\n    for name in names:\n        if isinstance(obj, dict) and name in obj and obj[name] is not None:\n            return obj[name]\n        if hasattr(obj, name):\n            value = getattr(obj, name)\n            if value is not None:\n                return value\n    return None\n\n\ndef tool_schema(tool):\n    schema = field(tool, "input_schema", "inputSchema")\n    if not isinstance(schema, dict):\n        raise RuntimeError("Cannot read tool input schema; verify against the running code.")\n    return schema\n\n\ndef payload_of(result):\n    """Prefer the JSON protocol payload over client-generated Python classes.\n\n    FastMCP\'s .data is a convenience decoding and may contain a generated\n    dataclass named Root. Preserve structured_content (or structuredContent)\n    as supplied; do not unwrap arbitrary \'result\' or \'root\' business fields.\n    """\n    if field(result, "is_error", "isError"):\n        raise MCPToolResultError(f"MCP error result: {plain(field(result, \'content\') or [])}")\n    value = field(result, "structured_content", "structuredContent")\n    if value is not None:\n        return plain(value)\n    parts = field(result, "content") or []\n    texts = [field(part, "text") for part in parts if field(part, "text") is not None]\n    if len(texts) == 1:\n        try:\n            return json.loads(texts[0])\n        except json.JSONDecodeError:\n            pass\n    value = field(result, "data")\n    if value is not None:\n        return plain(value)\n    return {"text_content": texts}\n\n\nWRAPPED_TOOLS_CACHE = {}\n\n\nasync def call(client, name, args, run_id=None):\n    start = time.monotonic()\n    event = {"at": utc(), "stage": STAGE, "run_id": run_id,\n             "tool": name, "arguments": args}\n    phase = "invoke"\n    try:\n        result = await client.call_tool(name, args)\n        phase = "normalize_response"\n        # Keep the protocol fields separately from FastMCP\'s optional decoded\n        # .data. A normalization error is not an upstream funding assertion.\n        event["response_type"] = type(result).__name__\n        event["decoded_data_type"] = type(field(result, "data")).__name__\n        event["protocol_response"] = {\n            "structured_content": plain(field(result, "structured_content", "structuredContent")),\n            "content": plain(field(result, "content") or []),\n            "is_error": bool(field(result, "is_error", "isError")),\n        }\n        value = payload_of(result)\n        # Protocol-level unwrap only: FastMCP wraps non-object (e.g. union)\n        # returns as {"result": ...} and marks the tool output schema with\n        # x-fastmcp-wrap-result. Unwrap only for tools declaring that marker.\n        if name not in WRAPPED_TOOLS_CACHE:\n            WRAPPED_TOOLS_CACHE.clear()\n            for t in await client.list_tools():\n                schema = field(t, "output_schema", "outputSchema") or {}\n                WRAPPED_TOOLS_CACHE[t.name] = bool(plain(schema).get("x-fastmcp-wrap-result"))\n        if WRAPPED_TOOLS_CACHE.get(name) and isinstance(value, dict) and set(value) == {"result"}:\n            value = value["result"]\n            event["protocol_result_unwrapped"] = True\n        event.update(status="SUCCESS", output=value)\n        return value\n    except Exception as exc:\n        origin = ("mcp_tool_result" if isinstance(exc, MCPToolResultError)\n                  else "evaluation_instrument" if phase == "normalize_response"\n                  else "mcp_call_or_transport")\n        event.update(status="ERROR", error_phase=phase, error_origin=origin,\n                     error_type=type(exc).__name__, error=str(exc))\n        raise\n    finally:\n        event["latency_s"] = time.monotonic() - start\n        append_event("tool_calls.jsonl", event)\n\n\nasync def inventory():\n    import httpx\n    from fastmcp import Client\n    async with Client(CFG["mcp_url"]) as client:\n        tools = await client.list_tools()\n        prompts = await client.list_prompts()\n        resources = await client.list_resources()\n        templates = await client.list_resource_templates()\n        guidelines = await client.read_resource("nih-reporter://response-guidelines")\n    observed = {"tools": len(tools), "prompts": len(prompts), "resources": len(resources)}\n    native = {"tools": plain(tools), "prompts": plain(prompts),\n              "resources": plain(resources), "resource_templates": plain(templates)}\n    save("inventory/native.json", native)\n    schemas = {t.name: tool_schema(t) for t in tools}\n    save("inventory/schemas.json", schemas)\n    save("inventory/response_guidelines.json", plain(guidelines))\n    async with httpx.AsyncClient(timeout=30) as http:\n        response = await http.get(CFG["mcp_url"].removesuffix("/mcp") + "/version")\n        response.raise_for_status()\n        version = response.json()\n    save("inventory/version_route.json", version)\n    require(observed == CFG["expected_inventory"], f"Inventory mismatch: {observed}")\n    require("clinical_trials_lookup" in schemas, "Real clinical_trials_lookup tool is absent.")\n    require(all(version.get(k) == n for k, n in observed.items()), "/version inventory mismatch.")\n    require(str(version.get("version", "")).removeprefix("v") == CFG["release_version"],\n            f"/version disagrees with release: {version.get(\'version\')}")\n    return {"status": "PASS", "observed": observed,\n            "resource_templates": len(templates), "version": version,\n            "scope": "Fresh local HTTP MCP process; not deployed-app verification."}\n\n\ndef independent_ledger(records, basis):\n    """Independent reducer: no imports from the production aggregation code."""\n    require(basis in {"whole_award", "all_records"}, "Unknown funding basis")\n    seen, unique, ledger, conflicts = {}, [], [], []\n    total = Decimal("0")\n    for index, row in enumerate(records):\n        aid = row.get("appl_id")\n        item = {"row_index": index, "appl_id": aid,\n                "core_project_num": row.get("core_project_num"),\n                "fiscal_year": row.get("fiscal_year"),\n                "subproject_id": row.get("subproject_id"),\n                "award_amount": row.get("award_amount"),\n                "amount_counted": "0", "decision": None}\n        if aid is not None and aid in seen:\n            previous = seen[aid]\n            if any(previous.get(k) != row.get(k)\n                   for k in ("award_amount", "fiscal_year", "subproject_id")):\n                conflicts.append(aid)\n            item["decision"] = "duplicate_appl_id"\n        else:\n            if aid is not None:\n                seen[aid] = row\n            unique.append(row)\n            if basis == "whole_award" and row.get("subproject_id") is not None:\n                item["decision"] = "excluded_subproject"\n            elif row.get("award_amount") is None:\n                item["decision"] = "missing_amount"\n            else:\n                amount = Decimal(str(row["award_amount"]))\n                total += amount\n                item.update(decision="included", amount_counted=str(amount))\n        ledger.append(item)\n    parents = {(r.get("core_project_num"), r.get("fiscal_year")) for r in unique\n               if r.get("subproject_id") is None}\n    orphan = [r.get("appl_id") for r in unique\n              if r.get("subproject_id") is not None and r.get("core_project_num")\n              and (r.get("core_project_num"), r.get("fiscal_year")) not in parents]\n    unidentified = [r["row_index"] for r in ledger if r["appl_id"] is None]\n    counts = Counter(r["decision"] for r in ledger)\n    return {"basis": basis, "total": str(total.quantize(Decimal("0.01"))),\n            "raw_count": len(records), "counts": dict(counts),\n            "contributing_ids": [r["appl_id"] for r in ledger\n                                 if r["decision"] == "included" and r["appl_id"] is not None],\n            "conflicting_ids": sorted(set(conflicts)),\n            "orphan_child_ids": orphan if basis == "whole_award" else [],\n            "rows_without_appl_id": unidentified, "ledger": ledger}\n\n\ndef save_ledger(prefix, result):\n    save(prefix + ".json", result)\n    path = ROOT / (prefix + ".csv")\n    rows = result["ledger"]\n    if rows:\n        with path.open("w", newline="", encoding="utf-8") as f:\n            writer = csv.DictWriter(f, fieldnames=list(rows[0]))\n            writer.writeheader()\n            writer.writerows(rows)\n\n\nasync def replay_bundle(records, criteria, arguments, total_matching, captured_payloads=None):\n    """Exercise the actual MCP tool with frozen data at the raw-fetch boundary."""\n    from fastmcp import Client\n    from nih_reporter_mcp.app import mcp\n    from nih_reporter_mcp.config import PROJECTS_SEARCH\n    from nih_reporter_mcp.models import FundingSummary\n    from nih_reporter_mcp.tools.evidence_bundle import EvidenceBundle\n    requests = []\n    async def replay(url, payload):\n        require(url == PROJECTS_SEARCH, f"Unexpected replay URL: {url}")\n        require(payload.get("criteria") == criteria, "Tool query differs from captured criteria.")\n        offset, limit = payload.get("offset", 0), payload.get("limit", 500)\n        if captured_payloads is not None:\n            require(any(p == payload for p in captured_payloads),\n                    "Tool payload not present in captured requests: verify against the running code.")\n        requests.append(copy.deepcopy(payload))\n        return {"meta": {"total": total_matching, "offset": offset, "limit": limit},\n                "results": copy.deepcopy(records[offset:offset + limit])}\n    # Unexpected HTTP requests must fail: replay may use only the supplied records.\n    async def deny_network(*args, **kwargs):\n        raise RuntimeError("Network attempted during frozen replay.")\n    with patch("nih_reporter_mcp.utils.post_json", new=replay), \\\n         patch("httpx.AsyncClient.send", new=deny_network), \\\n         patch("httpx.Client.send", side_effect=RuntimeError("Network attempted during replay.")):\n        async with Client(mcp) as client:\n            bundle = await call(client, "export_evidence_bundle", arguments)\n    EvidenceBundle.model_validate(bundle)\n    FundingSummary.model_validate(bundle["summary"])\n    return bundle, requests\n\n\ndef compare_bundle(bundle, oracle, total_matching):\n    summary, prov = bundle["summary"], bundle["provenance"]\n    require(Decimal(str(summary["total_award_amount_usd"])) == Decimal(oracle["total"]),\n            "Independent arithmetic and production MCP output differ.")\n    require(prov["records_aggregated"] == oracle["counts"].get("included", 0),\n            "Contributing count mismatch.")\n    require(summary["total_matching_projects"] == total_matching, "Raw-count mismatch.")\n    expected_ids = oracle["contributing_ids"][:500]\n    require(prov["source_record_ids"] == expected_ids, "Contributing ID sequence mismatch.")\n    require(prov["deduped_count"] == oracle["counts"].get("duplicate_appl_id", 0),\n            "Duplicate-count mismatch.")\n    require(prov["subproject_excluded_count"] == oracle["counts"].get("excluded_subproject", 0),\n            "Subproject-count mismatch.")\n    require(prov["records_with_missing_amount"] == oracle["counts"].get("missing_amount", 0),\n            "Null-amount count mismatch.")\n    require(set(prov["id_conflicts"]) == set(oracle["conflicting_ids"]), "Conflict mismatch.")\n    return True\n\n\nasync def fixtures():\n    from nih_reporter_mcp.models import FundingSummary\n    from nih_reporter_mcp.tools.evidence_bundle import EvidenceBundle\n    from nih_reporter_mcp.utils import aggregate_funding\n    reports = []\n    for folder in ("demonstration_fixture", "demonstration_fixture_double_counting"):\n        base = Path(CFG["repo_dir"]) / "docs" / folder\n        raw_path = base / "raw_records.json"\n        recorded_hash = (base / "raw_records.sha256").read_text().split()[0]\n        actual_hash = sha(raw_path)\n        require(actual_hash == recorded_hash, f"{folder}: historical raw hash mismatch.")\n        raw = json.loads(raw_path.read_text())\n        records = raw["results"]\n        report = {"fixture": folder, "raw_sha256": actual_hash,\n                  "historical_retrieved_at": raw.get("retrieval_timestamp_utc", raw.get("retrieved_at")),\n                  "replay_at": utc(), "comparisons": {}}\n        for basis in ("whole_award", "all_records"):\n            oracle = independent_ledger(records, basis)\n            save_ledger(f"fixtures/{folder}/{basis}_ledger", oracle)\n            production = aggregate_funding(copy.deepcopy(records), funding_basis=basis)\n            require(Decimal(str(production.total_award)) == Decimal(oracle["total"]),\n                    f"{folder}/{basis}: production reducer mismatch.")\n            require(production.contributing_count == oracle["counts"].get("included", 0),\n                    "Production contributing count differs.")\n            require(set(production.orphan_children) == set(oracle["orphan_child_ids"]),\n                    "Production orphan detection differs.")\n            report["comparisons"][basis] = {k: v for k, v in oracle.items() if k != "ledger"}\n        if folder == "demonstration_fixture":\n            stored = json.loads((base / "tool_output.json").read_text())\n            # Validate the actual envelope; do not repair or coerce the historical artifact.\n            EvidenceBundle.model_validate(stored)\n            FundingSummary.model_validate(stored["summary"])\n            criteria = raw["request_payload"]["criteria"]\n            arguments = {**CFG["cohort"], "label": "Historical fixture replay"}\n            replayed, requests = await replay_bundle(records, criteria, arguments, raw["meta"]["total"])\n            save(f"fixtures/{folder}/production_replay.json",\n                 {"mode": "FROZEN_OFFLINE_REPLAY", "source_acquired_at": report["historical_retrieved_at"],\n                  "replay_at": utc(), "tool_return_value": replayed, "replay_requests": requests})\n            whole = independent_ledger(records, "whole_award")\n            compare_bundle(replayed, whole, raw["meta"]["total"])\n            require(Decimal(str(stored["summary"]["total_award_amount_usd"])) == Decimal(whole["total"]),\n                    "Historical response total does not reconcile.")\n            require(set(stored["provenance"]["source_record_ids"]) == set(whole["contributing_ids"]),\n                    "Historical contributing IDs do not reconcile.")\n            report["stored_response_retrieved_at"] = stored["provenance"]["retrieved_at"]\n            report["scope"] = ("Raw fixture projection verifies funding arithmetic and IDs, not "\n                               "organization/IC distinct counts. Orphan exclusion is not parent-child "\n                               "double-counting proof. Historical and replay timestamps are distinct.")\n        else:\n            whole, all_rows = report["comparisons"]["whole_award"], report["comparisons"]["all_records"]\n            require(whole["counts"].get("excluded_subproject", 0) > 0, "No subprojects in fixture.")\n            require(not whole["orphan_child_ids"], "Parent-child fixture contains unresolved orphans.")\n            report["basis_difference"] = str(Decimal(all_rows["total"]) - Decimal(whole["total"]))\n            report["scope"] = "Frozen parent-plus-component fixture; production reducer comparison."\n        reports.append(report)\n    return {"status": "PASS", "fixtures": reports}\n\n\nasync def live_capture():\n    import httpx\n    from nih_reporter_mcp.utils import AGG_FIELDS\n    criteria = CFG["query_criteria"]\n    base = {"criteria": criteria, "include_fields": AGG_FIELDS,\n            "sort_field": "award_amount", "sort_order": "desc"}\n    records, payloads, observations = [], [], []\n    offset, total = 0, None\n    began = utc()\n    async with httpx.AsyncClient(timeout=90, follow_redirects=False, trust_env=False) as client:\n        while total is None or offset < min(total, 10000):\n            payload = {**base, "offset": offset, "limit": 500}\n            response = None\n            for attempt in range(3):\n                response = await client.post(CFG["reporter_url"], json=payload)\n                raw_name = f"live/raw/page_{offset:05d}_attempt_{attempt + 1}.json"\n                raw_path = ROOT / raw_name\n                raw_path.parent.mkdir(parents=True, exist_ok=True)\n                raw_path.write_bytes(response.content)\n                observations.append({"at": utc(), "request": payload, "http_status": response.status_code,\n                                     "raw_path": raw_name, "sha256": sha(raw_path)})\n                save("live/requests.json", observations)\n                if response.status_code != 429:\n                    break\n                await asyncio.sleep(min(2 ** (attempt + 1), 8))\n            response.raise_for_status()\n            data = response.json()\n            current_total = data.get("meta", {}).get("total")\n            require(isinstance(current_total, int), "RePORTER omitted an integer meta.total.")\n            if total is None:\n                total = current_total\n            require(total == current_total, "Upstream count changed between pages; capture is unstable.")\n            batch = data.get("results", [])\n            require(isinstance(batch, list), "Malformed RePORTER result list.")\n            require(batch or total == 0, "Unexpected empty page before end of cohort.")\n            payloads.append(payload)\n            records.extend(batch)\n            offset += 500\n            if total == 0:\n                break\n            await asyncio.sleep(1.1)\n    distinct = {r.get("appl_id") for r in records if r.get("appl_id") is not None}\n    complete = len(distinct) >= total if distinct else len(records) >= total\n    save("live/captured_records.json", {"criteria": criteria, "total_matching": total,\n         "complete": complete, "capture_started_at": began, "capture_ended_at": utc(),\n         "results": records})\n    require(complete, "Demo cohort exceeded the retrieval ceiling; narrow it before claiming exactness.")\n    comparisons = {}\n    for basis in ("whole_award", "all_records"):\n        oracle = independent_ledger(records, basis)\n        save_ledger(f"live/{basis}_ledger", oracle)\n        bundle, requests = await replay_bundle(\n            records, criteria, {**CFG["cohort"], "funding_basis": basis,\n                                "label": "Captured-live-data replay"}, total, payloads)\n        save(f"live/{basis}_bundle.json",\n             {"mode": "LIVE_SOURCE_CAPTURE_THEN_OFFLINE_MCP_REPLAY",\n              "source_capture_started_at": began, "source_capture_ended_at": observations[-1]["at"],\n              "replayed_at": utc(), "tool_return_value": bundle, "replay_requests": requests})\n        compare_bundle(bundle, oracle, total)\n        comparisons[basis] = {k: v for k, v in oracle.items() if k != "ledger"}\n        comparisons[basis]["tool_note"] = bundle["summary"]["note"]\n    w = Decimal(comparisons["whole_award"]["total"])\n    a = Decimal(comparisons["all_records"]["total"])\n    return {"status": "PASS", "mode": "LIVE_CAPTURE_PLUS_OFFLINE_MCP_REPLAY",\n            "query_criteria": criteria, "total_matching": total, "complete": complete,\n            "comparisons": comparisons,\n            "basis_difference_percent": str((a - w) * 100 / w) if w else None,\n            "limits": [\n                "Same captured records are used for both bases and independent reconciliation.",\n                "This is not a direct deployed-app test or an atomic NIH database snapshot.",\n                "Raw byte hashes establish identity, not scientific attribution.",\n                "Orphans, missing amounts, conflicting IDs and missing IDs limit investment claims.",\n                "The topic is a text-search cohort, not an official category or program roster."]}\n\n\nasync def smoke():\n    from fastmcp import Client\n    checks = []\n    async def check(name, fn):\n        try:\n            detail = await fn()\n            checks.append({"name": name, "status": "PASS", "detail": detail})\n        except Exception as exc:\n            checks.append({"name": name, "status": "FAIL", "error_type": type(exc).__name__,\n                           "detail": str(exc)})\n    async with Client(CFG["mcp_url"]) as client:\n        async def search_check():\n            d = await call(client, "search_funded_projects",\n                           {"topic": "CRISPR sickle cell", "fiscal_years": [2024], "limit": 5})\n            require(d["returned"] == len(d["projects"]) <= 5, "Search cardinality mismatch.")\n            return {"returned": d["returned"], "total_matching": d["total_matching_projects"]}\n        await check("Search response and requested limit", search_check)\n\n        async def evidence_check():\n            d = await call(client, "export_evidence_bundle", CFG["cohort"])\n            s, p = d["summary"], d["provenance"]\n            require(s["funding_basis"] == "whole_award", "Wrong funding basis.")\n            require(p["records_aggregated"] == s["projects_aggregated"], "Contributing count mismatch.")\n            require(len(p["source_record_ids"]) <= min(500, p["records_aggregated"]), "ID-cap mismatch.")\n            require(len(p["source_record_ids"]) == len(set(p["source_record_ids"])),\n                    "Contributing IDs contain duplicates.")\n            accounted = p["records_aggregated"] + p["deduped_count"] + p["subproject_excluded_count"] + p["records_with_missing_amount"]\n            require(bool(d["reproduction_note"]), "No reproduction note.")\n            return {"summary": s, "ids_shown": len(p["source_record_ids"]),\n                    "implied_fetched_rows_including_duplicates": accounted,\n                    "scope": "Live structure/accounting; fetched rows may exceed meta.total because of duplicates. "\n                             "Independent arithmetic and full ID accounting are in the captured-data demo."}\n        await check("Evidence bundle, count accounting and capped IDs", evidence_check)\n\n        async def org_check():\n            name = "BOSTON UNIVERSITY MEDICAL CAMPUS"\n            d = await call(client, "organization_funding_profile",\n                           {"organization_name": name, "fiscal_years": [2024], "exact_match": True})\n            require(all(n.casefold() == name.casefold() for n in d["matched_org_names"]),\n                    "Exact organization query returned another organization name.")\n            return {"matched_org_names": d["matched_org_names"], "raw_count": d["total_matching_projects"],\n                    "scope": "No fixed historical count; an empty match is not proof of no funding."}\n        await check("Exact organization-name isolation", org_check)\n\n        async def comparison_check():\n            values = {}\n            for key, args in [("category", {"spending_category": "Long COVID"}),\n                              ("text", {"topic": "long covid"})]:\n                values[key] = await call(client, "summarize_funding_by_topic",\n                    {**args, "fiscal_years": [2023], "funding_basis": "whole_award",\n                     "include_provenance": True})\n            require(values["category"]["provenance"]["query_criteria"] !=\n                    values["text"]["provenance"]["query_criteria"], "Cohort definitions unexpectedly identical.")\n            save("smoke/category_vs_text.json", values)\n            return {k: {"amount": v["total_award_amount_usd"], "complete": v["complete"],\n                        "basis": v["funding_basis"], "note": v["note"]} for k, v in values.items()}\n        await check("Category versus text: execute both defined cohorts", comparison_check)\n\n        async def validation_check():\n            try:\n                await call(client, "search_funded_projects", {"topic": ""})\n            except Exception as exc:\n                message = str(exc).lower()\n                require("provide at least" in message or "empty" in message,\n                        "Failure was not the expected input validation contract.")\n                return "Missing-query input rejected; upstream 403 behavior is covered by mocked CI tests."\n            raise AssertionError("Invalid input was accepted.")\n        await check("Missing-query validation", validation_check)\n\n        async def trend_check():\n            d = await call(client, "funding_trend_by_year",\n                           {"spending_category": "Long COVID", "start_year": 2022,\n                            "end_year": 2023, "funding_basis": "whole_award"})\n            require(sorted(y["fiscal_year"] for y in d["years"]) == [2022, 2023],\n                    "Returned years differ from requested years.")\n            return {"years": d["years"], "note": d["note"],\n                    "limit": "Zero matches are not evidence of zero NIH investment."}\n        await check("Trend returns every requested year", trend_check)\n\n        async def overlap_check():\n            args = {"last_name": "Au", "first_name": "Rhoda"}\n            a = await call(client, "check_investigator_overlap", args)\n            b = await call(client, "check_investigator_overlap",\n                           {**args, "exclude_project_num": "U19AG068753"})\n            require(isinstance(a["total_active_awards"], int), "Invalid active-award count.")\n            require(0 <= b["total_active_awards"] <= a["total_active_awards"],\n                    "Exclusion increased the count.")\n            return {"before": a["total_active_awards"], "after": b["total_active_awards"],\n                    "strict_reduction_observed": b["total_active_awards"] < a["total_active_awards"],\n                    "limit": "If counts are equal, active inclusion of the named award was not established."}\n        await check("Overlap structure and non-increasing exclusion", overlap_check)\n\n        async def section_check():\n            d = await call(client, "funded_projects_by_study_section",\n                           {"topic": "CRISPR sickle cell", "fiscal_years": [2024]})\n            require(isinstance(d["sections"], list), "Invalid study-section list.")\n            require(all(s["project_count"] >= 0 for s in d["sections"]), "Negative section count.")\n            return {"sections": len(d["sections"]), "note": d["note"],\n                    "scope": "Structural check only, not independent reconciliation or review-workload measurement."}\n        await check("Funded-study-section structure", section_check)\n\n        async def recent_check():\n            d = await call(client, "search_recent_awards", {"limit": 5})\n            require(d["returned"] == len(d["awards"]) <= 5, "Recent-awards cardinality mismatch.")\n            require(all(a.get("reporter_url") for a in d["awards"]), "Missing RePORTER link.")\n            return {"returned": d["returned"], "note": d["note"]}\n        await check("Recently added records, not award-date recency", recent_check)\n\n        async def health_check():\n            positive = await call(client, "explain_health_topic", {"topic": "diabetes"})\n            negative = await call(client, "explain_health_topic", {"topic": "zzznotarealcondition999"})\n            require(positive["found"] and bool(positive.get("summary")), "No expected common-topic summary.")\n            require(negative["found"] is False, "Unexpected match for synthetic nonexistent condition.")\n            return {"positive_found": positive["found"], "negative_found": negative["found"]}\n        await check("MedlinePlus match and no-match", health_check)\n\n        async def repeated_check():\n            args = CFG["cohort"]\n            a = await call(client, "summarize_funding_by_topic", args)\n            b = await call(client, "summarize_funding_by_topic", args)\n            same = a["total_award_amount_usd"] == b["total_award_amount_usd"]\n            return {"same_observed_total": same, "first": a["total_award_amount_usd"],\n                    "second": b["total_award_amount_usd"],\n                    "scope": "Observation only: does not establish cache hits, a speedup, or source immutability."}\n        await check("Repeated-call consistency observation", repeated_check)\n\n        async def ct_check():\n            d = await call(client, "clinical_trials_lookup", {"core_project_num": "R01HL147914"})\n            ids = [t["nct_id"] for t in d["trials"]]\n            require(len(ids) == len(set(ids)) == d["distinct_trials"], "Trial dedup count mismatch.")\n            require(bool(d["boundary_note"]) and bool(d["dollar_attribution_note"]), "Missing linkage limits.")\n            require(all("total_award_amount_usd" not in t for t in d["trials"]), "Unexpected trial dollar attribution.")\n            return {"distinct_trials": d["distinct_trials"], "query_type": d["query_type"],\n                    "boundary_note": d["boundary_note"],\n                    "scope": "Live structural lookup; no fixed hit count. Structured linkage fixtures are covered by CI."}\n        await check("Real clinical_trials_lookup tool and attribution limits", ct_check)\n    counts = Counter(c["status"] for c in checks)\n    return {"status": "FAIL" if counts["FAIL"] else "PASS", "counts": dict(counts),\n            "checks": checks, "scope": "Notebook-defined live checks, not the tagged standalone smoke suite or CI.",\n            "not_covered": ["All-tools exhaustive behavior", "10K retrieval instrumentation",\n                            "Live upstream WAF behavior", "Cache hit/miss instrumentation"]}\n\n\ndef run_ci():\n    selectors = CFG["ci_selectors"]\n    repo = Path(CFG["repo_dir"])\n    workflow = (repo / ".github/workflows/ci.yml").read_text()\n    require(all(selector in workflow for selector in selectors), "CI selectors differ from pinned workflow.")\n    plugin = ROOT / "ci_outcomes_plugin.py"\n    plugin.write_text(\'\'\'import json, os\nfrom collections import Counter\nrows = {}\ncollection_errors = []\ndef pytest_runtest_logreport(report):\n    key = report.nodeid\n    if report.when == "call":\n        if hasattr(report, "wasxfail"):\n            rows[key] = "xfail" if report.skipped else "xpass"\n        else:\n            rows[key] = {"passed":"passed", "failed":"failed", "skipped":"skipped"}[report.outcome]\n    elif report.failed:\n        rows[key] = "error"\n    elif report.skipped:\n        rows[key] = "xfail" if hasattr(report, "wasxfail") else "skipped"\ndef pytest_collectreport(report):\n    if report.failed:\n        collection_errors.append(str(report.longrepr))\ndef pytest_sessionfinish(session, exitstatus):\n    all_ids = [item.nodeid for item in session.items]\n    for node in all_ids:\n        rows.setdefault(node, "not_run")\n    data = {"counts": dict(Counter(rows.values())), "outcomes": rows,\n            "collected_nodeids": all_ids, "collection_errors": collection_errors,\n            "exit_code": int(exitstatus)}\n    with open(os.environ["NIH_EVAL_CI_REPORT"], "w") as f:\n        json.dump(data, f, indent=2)\n\'\'\', encoding="utf-8")\n    report_path = ROOT / "ci/outcomes.json"\n    report_path.parent.mkdir(exist_ok=True)\n    env = {**os.environ, "NIH_EVAL_CI_REPORT": str(report_path),\n           "PYTHONPATH": str(ROOT) + os.pathsep + os.environ.get("PYTHONPATH", ""),\n           "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1"}\n    env.pop("PYTEST_ADDOPTS", None)\n    command = [sys.executable, "-S", "-m", "pytest", *selectors, "-v", "--tb=short",\n               "-p", "ci_outcomes_plugin", "-p", "no:cacheprovider",\n               "--junitxml=" + str(ROOT / "ci/junit.xml")]\n    with (ROOT / "ci/stdout.log").open("w") as f:\n        completed = subprocess.run(command, cwd=repo, env=env, stdout=f, stderr=subprocess.STDOUT)\n    observed = json.loads(report_path.read_text()) if report_path.exists() else {\n        "counts": {}, "collection_errors": ["No machine-readable report produced."]}\n    expected = {"passed": 155, "xfail": 2}\n    matches = observed["counts"] == expected and not observed["collection_errors"]\n    return {"status": "PASS" if completed.returncode == 0 and matches else "FAIL",\n            "expected": expected, "observed": observed, "command": command,\n            "exit_code": completed.returncode,\n            "scope": "Tagged CI selection with reporting plugin and plugin autoload disabled; one recorded Python runtime."}\n\n\ndef project_for_model(payload, max_chars=12000):\n    """Keep every scalar/caveat, including nested summary/provenance. Soft budget."""\n    omitted = {}\n    def trim(obj, path=""):\n        if isinstance(obj, dict):\n            return {k: trim(v, path + "/" + k) for k, v in obj.items()}\n        if isinstance(obj, list):\n            if len(obj) > 5:\n                omitted[path or "/"] = {"original_count": len(obj), "shown_count": 5}\n            return [trim(v, path + f"/{i}") for i, v in enumerate(obj[:5])]\n        return obj\n    value = plain(payload)\n    text = json.dumps(value, ensure_ascii=False)\n    if len(text) <= max_chars:\n        return text\n    result = {"data": trim(value), "_presentation": {\n        "truncated_arrays": omitted, "full_output_saved": True,\n        "character_budget_is_soft": True,\n        "warning": "Lists may be abbreviated. Do not rank or enumerate omitted records."}}\n    text = json.dumps(result, ensure_ascii=False)\n    if len(text) > max_chars:\n        result["_presentation"]["budget_exceeded_to_preserve_caveats"] = True\n    return json.dumps(result, ensure_ascii=False)\n\n\ndef validate_arguments(tool_name, args, schemas):\n    from jsonschema.validators import validator_for\n    if not isinstance(args, dict):\n        return {"valid": False, "errors": ["Tool arguments must be a JSON object."]}\n    schema = schemas.get(tool_name)\n    if schema is None:\n        return {"valid": False, "errors": [f"Unknown tool: {tool_name}"]}\n    validator = validator_for(schema)\n    validator.check_schema(schema)\n    errors = [e.message for e in validator(schema).iter_errors(args)]\n    return {"valid": not errors, "errors": errors,\n            "scope": "JSON Schema only; tool-specific runtime constraints are scored separately."}\n\n\ndef runtime_preflight(tool, args):\n    """Only explicit, source-inspected cross-field contracts; never claim completeness."""\n    errors = []\n    if tool == "get_project_by_id" and not (args.get("appl_id") or args.get("core_project_num")):\n        errors.append("Provide either appl_id or core_project_num.")\n    if tool == "clinical_trials_lookup":\n        if bool(args.get("core_project_num")) == bool(args.get("condition")):\n            errors.append("Provide exactly one of core_project_num or condition.")\n    return {"known_contracts_pass": not errors, "errors": errors,\n            "complete_runtime_validation": False}\n\n\nSYSTEM_PROMPT = """You are a research-funding analyst using NIH RePORTER MCP tools.\nUse tools for specific funding facts. State the query, fiscal year, counting unit,\nfunding basis and completeness. Use whole_award for investment totals. Preserve\nmissing-amount, duplicate-conflict, orphan-child and truncation limitations.\nZero matches do not establish zero investment or no relationship.\nCategory-tagged RePORTER aggregation is not automatically a published RCDC expenditure.\nFor a methods comparison, execute both defined queries using the same funding basis.\nDo not rank omitted records or count component records as distinct parent awards.\nIf the tools cannot answer a request, state the unsupported capability.\nRecently added records describe database recency, not award-date recency.\nGrant-output associations do not establish causation or fiscal-year attribution.\nConsult the supplied release response guidelines, subject to these evidence boundaries."""\n\n\nasync def model_context():\n    from fastmcp import Client\n    from openai import AsyncOpenAI\n    host = os.environ.get("NIH_EVAL_DB_HOST", "").rstrip("/")\n    token = os.environ.get("NIH_EVAL_DB_TOKEN", "")\n    require(host.startswith("https://") and token, "Model section NOT RUN: credentials unavailable.")\n    async with Client(CFG["mcp_url"]) as client:\n        tools = await client.list_tools()\n        resources = await client.read_resource("nih-reporter://response-guidelines")\n    schemas = {t.name: tool_schema(t) for t in tools}\n    provider = [{"type": "function", "function": {"name": t.name,\n                "description": t.description or "", "parameters": schemas[t.name]}} for t in tools]\n    save("agents/model_facing_schemas.json", provider)\n    instructions = SYSTEM_PROMPT + "\\nRelease response guidelines:\\n" + json.dumps(plain(resources))\n    save("agents/system_prompt.json", {"text": instructions})\n    llm = AsyncOpenAI(api_key=token, base_url=host + "/serving-endpoints/",\n                      timeout=90, max_retries=0)\n    return llm, provider, schemas, instructions\n\n\nasync def model_request(llm, model, messages, tools, run_id):\n    started = time.monotonic()\n    request = {"model": model, "messages": messages, "tools": tools,\n               "tool_choice": "auto", **CFG["model_parameters"]}\n    event = {"at": utc(), "run_id": run_id, "stage": STAGE, "request": request,\n             "decoding": {"sent": CFG["model_parameters"],\n                          "unspecified_settings": "provider defaults; not claimed fixed"}}\n    try:\n        response = await llm.chat.completions.create(**request)\n        event.update(status="SUCCESS", response=plain(response))\n        return response\n    except Exception as exc:\n        event.update(status="ENDPOINT_ERROR", error_type=type(exc).__name__, error=str(exc))\n        raise\n    finally:\n        event["latency_s"] = time.monotonic() - started\n        append_event("agents/traces.jsonl", event)\n\n\nasync def run_agent(question, model, run_id):\n    from fastmcp import Client\n    llm, tools, schemas, instructions = await model_context()\n    messages = [{"role": "system", "content": instructions}, {"role": "user", "content": question}]\n    calls, model_rounds = [], 0\n    try:\n        async with Client(CFG["mcp_url"]) as client:\n            for model_rounds in range(1, 7):\n                try:\n                    response = await model_request(llm, model, messages, tools, run_id)\n                except Exception as exc:\n                    return {"status": "ENDPOINT_ERROR", "error": str(exc), "calls": calls,\n                            "model_rounds": model_rounds, "answer": None}\n                msg = response.choices[0].message\n                if not msg.tool_calls:\n                    answer = msg.content or ""\n                    status = "COMPLETED" if answer.strip() else "EMPTY_ANSWER"\n                    return {"status": status, "calls": calls, "answer": answer,\n                            "model_rounds": model_rounds,\n                            "answer_correctness": "NOT_SCORED; human review required"}\n                messages.append({"role": "assistant", "content": msg.content,\n                                 "tool_calls": [plain(tc) for tc in msg.tool_calls]})\n                for tc in msg.tool_calls:\n                    record = {"tool": tc.function.name, "raw_arguments": tc.function.arguments}\n                    try:\n                        args = json.loads(tc.function.arguments)\n                        check = validate_arguments(tc.function.name, args, schemas)\n                        require(check["valid"], "Schema-invalid arguments: " + str(check["errors"]))\n                        guard = runtime_preflight(tc.function.name, args)\n                        require(guard["known_contracts_pass"], str(guard["errors"]))\n                        result = await call(client, tc.function.name, args, run_id)\n                        visible = project_for_model(result)\n                        record.update(status="SUCCESS", arguments=args)\n                    except Exception as exc:\n                        visible = json.dumps({"error": str(exc), "status": "TOOL_CALL_FAILED"})\n                        record.update(status="ERROR", error=str(exc))\n                    calls.append(record)\n                    append_event("agents/model_visible_outputs.jsonl",\n                                 {"run_id": run_id, "tool_call_id": tc.id,\n                                  "tool": tc.function.name, "model_visible_output": visible})\n                    messages.append({"role": "tool", "tool_call_id": tc.id, "content": visible})\n            return {"status": "MAX_ROUNDS", "calls": calls, "model_rounds": model_rounds, "answer": None}\n    finally:\n        await llm.close()\n\n\nasync def connection():\n    import httpx\n    host = os.environ.get("NIH_EVAL_DB_HOST", "").rstrip("/")\n    token = os.environ.get("NIH_EVAL_DB_TOKEN", "")\n    require(host and token, "Model section NOT RUN: credentials unavailable.")\n    # Discovery permissions can differ from invocation permissions; report separately.\n    discovery = {}\n    async with httpx.AsyncClient(timeout=30) as http:\n        try:\n            response = await http.get(host + "/api/2.0/serving-endpoints",\n                                      headers={"Authorization": "Bearer " + token})\n            response.raise_for_status()\n            names = [e["name"] for e in response.json().get("endpoints", [])]\n            discovery = {"status": "SUCCESS", "names": names}\n        except Exception as exc:\n            discovery = {"status": "UNAVAILABLE", "error": str(exc),\n                         "note": "Invocation of explicitly configured endpoints is checked separately."}\n    save("agents/endpoint_discovery.json", discovery)\n    llm, _, _, _ = await model_context()\n    probe = [{"type": "function", "function": {"name": "evaluation_probe",\n              "description": "Return the supplied probe value.", "parameters": {\n                  "type": "object", "properties": {"value": {"type": "string"}},\n                  "required": ["value"], "additionalProperties": False}}}]\n    results = []\n    try:\n        for model in dict.fromkeys([CFG["demo_model"], *CFG["eval_models"]]):\n            try:\n                response = await model_request(llm, model,\n                    [{"role": "user", "content": "Call evaluation_probe with value=\'ok\'."}],\n                    probe, "preflight:" + model)\n                selections = response.choices[0].message.tool_calls or []\n                observed = any(t.function.name == "evaluation_probe" for t in selections)\n                results.append({"model": model, "status": "PASS" if observed else "NO_TOOL_CALL"})\n            except Exception as exc:\n                results.append({"model": model, "status": "ENDPOINT_ERROR", "error": str(exc)})\n    finally:\n        await llm.close()\n    return {"status": "PASS" if all(r["status"] == "PASS" for r in results) else "BLOCKED",\n            "discovery": discovery, "endpoints": results,\n            "scope": "Observed tool-calling preflight, not a claim about model families."}\n\n\nDEMO_QUESTIONS = [\n    "For CRISPR sickle cell research in FY2024, export the evidence bundle using whole_award. "\n    "State the matching-record and contributing-record counts, funding total and limitations.",\n    "Compare the Long COVID RCDC-tagged cohort with a free-text \'long covid\' cohort in FY2023, "\n    "using whole_award for both. State both queries, completeness and limits.",\n    "What relationships does clinical_trials_lookup report for R01HL147914? "\n    "Distinguish registered associations from outcomes and do not attribute dollars to trials.",\n]\n\n\nasync def demo():\n    runs = []\n    for i, question in enumerate(DEMO_QUESTIONS, 1):\n        result = await run_agent(question, CFG["demo_model"], f"demo:{i}")\n        result["question"] = question\n        if i == 2:\n            successful = [r.get("arguments", {}) for r in result["calls"]\n                          if r["status"] == "SUCCESS" and r["tool"] == "summarize_funding_by_topic"]\n            result["comparison_queries_observed"] = (\n                any(a.get("spending_category") for a in successful) and\n                any(a.get("topic") for a in successful))\n        runs.append(result)\n    return {"status": "COMPLETED" if all(r["status"] == "COMPLETED" for r in runs) else "INCOMPLETE",\n            "runs": runs, "scope": "Recorded agent demonstrations; final-answer accuracy is not automatically verified."}\n\n\nasync def efficiency():\n    question = ("Find five highest-amount RePORTER records matching Alzheimer\'s biomarkers, "\n                "FY2024, recipient state MA. Label the record grain and do not call them "\n                "five distinct whole awards unless that is established.")\n    runs = [await run_agent(question, CFG["demo_model"], f"efficiency:{i}") for i in range(3)]\n    counts = []\n    for result in runs:\n        success = result["status"] == "COMPLETED" and any(\n            r["tool"] == "search_funded_projects" and r["status"] == "SUCCESS" for r in result["calls"])\n        success = success and all(r["status"] == "SUCCESS" for r in result["calls"])\n        result["eligible_for_efficiency"] = success\n        if success:\n            counts.append(len(result["calls"]))\n    return {"status": "COMPLETED" if len(counts) == len(runs) else "INCOMPLETE",\n            "attempted_runs": len(runs), "eligible_runs": len(counts),\n            "failed_or_ineligible_runs": len(runs) - len(counts), "tool_calls_per_eligible_run": counts,\n            "mean_calls_conditional_on_eligibility": sum(counts) / len(counts) if counts else None,\n            "runs": runs, "limit": "Observed call counts; not model-round counts or proof of answer correctness."}\n\n\n# Legacy questions retained as a named regression slice, not a complete server benchmark.\nCASES = [\n    ("Q1", "How much did NIH spend on CRISPR sickle cell research in FY2024?", "summarize_funding_by_topic"),\n    ("Q2", "What are the five biggest Alzheimer\'s biomarker grants in Massachusetts in FY2024?", "search_funded_projects"),\n    ("Q3", "Is NIH funding for long COVID research growing or shrinking between 2021 and 2023?", "funding_trend_by_year"),\n    ("Q4", "Show me the funding profile for Johns Hopkins University in FY2024.", "organization_funding_profile"),\n    ("Q5", "What grants does Francis Collins currently hold?", "investigator_portfolio"),\n    ("Q6", "Look up the details of NIH project application 10876342.", "get_project_by_id"),\n    ("Q7", "What papers have been published from grant R01HL169761?", "list_publications_for_project"),\n    ("Q8", "What are the most recent NIH awards in the last week?", "search_recent_awards"),\n]\n\n\nasync def first_call_trial(llm, tools, schemas, instructions, case, model, run_id):\n    cid, question, expected = case\n    trial = {"case_id": cid, "model": model, "question": question,\n             "expected_first_tool": expected, "run_id": run_id}\n    try:\n        response = await model_request(llm, model,\n            [{"role": "system", "content": instructions}, {"role": "user", "content": question}],\n            tools, run_id)\n    except Exception as exc:\n        trial.update(status="ENDPOINT_ERROR", error=str(exc), selection_correct=False)\n        return trial\n    try:\n        selected = response.choices[0].message.tool_calls or []\n        if not selected:\n            trial.update(status="NO_TOOL_CALL", selection_correct=False)\n            return trial\n        tc = selected[0]\n        trial.update(status="SUCCESS", selected_tool=tc.function.name,\n                     selection_correct=tc.function.name == expected, additional_calls=len(selected) - 1,\n                     raw_arguments=tc.function.arguments)\n        try:\n            args = json.loads(tc.function.arguments)\n        except json.JSONDecodeError:\n            trial.update(schema_valid=False, validation_errors=["Malformed JSON arguments"],\n                         runtime_admissibility="NOT_TESTED")\n            return trial\n        validation = validate_arguments(tc.function.name, args, schemas)\n        trial.update(arguments=args, schema_valid=validation["valid"],\n                     validation_errors=validation["errors"])\n        if validation["valid"]:\n            preflight = runtime_preflight(tc.function.name, args)\n            trial["known_runtime_contracts"] = preflight\n            trial["runtime_admissibility"] = "KNOWN_CONTRACT_REJECTED" if preflight["errors"] else "NOT_FULLY_TESTED"\n        else:\n            trial["runtime_admissibility"] = "NOT_TESTED"\n        # Runtime invocation is deliberately not inferred from schema validity.\n        return trial\n    except Exception as exc:\n        trial.update(status="EVALUATION_ERROR", error=str(exc))\n        trial.setdefault("selection_correct", False)\n        return trial\n\n\nasync def selection():\n    llm, tools, schemas, instructions = await model_context()\n    rows = []\n    save("agents/preregistered_cases.json", {"cases": CASES,\n         "scoring": "First tool name only. Q8 does not test ability to enforce a last-week date filter.",\n         "scope": "Legacy eight-question regression slice; no claim of comprehensive capability."})\n    try:\n        for model in CFG["eval_models"]:\n            for case in CASES:\n                rows.append(await first_call_trial(llm, tools, schemas, instructions, case, model,\n                                                  f"selection:{model}:{case[0]}"))\n                save("agents/selection_trials.json", rows)\n                await asyncio.sleep(0.3)\n    finally:\n        await llm.close()\n    successful_responses = [r for r in rows if r["status"] != "ENDPOINT_ERROR"]\n    calls = [r for r in rows if "selected_tool" in r]\n    evaluation_errors = sum(r["status"] == "EVALUATION_ERROR" for r in rows)\n    correct = sum(r["selection_correct"] for r in rows)\n    unanimous = sum(all(any(r["case_id"] == case[0] and r["model"] == model\n                             and r["selection_correct"] for r in rows)\n                        for model in CFG["eval_models"]) for case in CASES)\n    return {"status": "COMPLETED" if len(successful_responses) == len(rows) and not evaluation_errors else "INCOMPLETE",\n            "planned_trials": len(CASES) * len(CFG["eval_models"]), "recorded_trials": len(rows),\n            "endpoint_errors": len(rows) - len(successful_responses),\n            "evaluation_errors": evaluation_errors,\n            "selection_correct": correct, "selection_denominator_all_planned": len(rows),\n            "selection_denominator_completed_responses": len(successful_responses),\n            "schema_valid_calls": sum(bool(r.get("schema_valid")) for r in calls),\n            "schema_validity_denominator_generated_first_calls": len(calls),\n            "known_runtime_rejections": sum(r.get("runtime_admissibility") == "KNOWN_CONTRACT_REJECTED" for r in rows),\n            "runtime_admissibility": "Not comprehensively measured; no runtime-validity percentage is claimed.",\n            "unanimous_correct_cases": unanimous, "unanimous_denominator": len(CASES),\n            "trials": rows}\n\n\nasync def stability():\n    llm, tools, schemas, instructions = await model_context()\n    results = []\n    try:\n        for case in CASES[:3]:\n            trials = [await first_call_trial(llm, tools, schemas, instructions, case,\n                      CFG["demo_model"], f"stability:{case[0]}:{i}") for i in range(5)]\n            success = all(t["status"] == "SUCCESS" for t in trials)\n            names = [t.get("selected_tool") for t in trials]\n            results.append({"case_id": case[0], "all_invocations_succeeded": success,\n                            "consistent_selection": success and len(set(names)) == 1,\n                            "all_selections_correct": success and all(t["selection_correct"] for t in trials),\n                            "trials": trials})\n    finally:\n        await llm.close()\n    return {"status": "COMPLETED" if all(r["all_invocations_succeeded"] for r in results) else "INCOMPLETE",\n            "consistent_cases": sum(r["consistent_selection"] for r in results),\n            "correct_and_consistent_cases": sum(r["consistent_selection"] and r["all_selections_correct"] for r in results),\n            "case_count": len(results), "repeats_per_case": 5, "cases": results,\n            "limit": "Observed consistency in a small sample, not general determinism."}\n\n\ndef main():\n    global ROOT, CFG, STAGE\n    parser = argparse.ArgumentParser()\n    parser.add_argument("action")\n    parser.add_argument("--config", required=True)\n    args = parser.parse_args()\n    CFG = json.loads(Path(args.config).read_text())\n    ROOT, STAGE = Path(CFG["artifact_dir"]), args.action\n    ROOT.mkdir(parents=True, exist_ok=True)\n    started = utc()\n    try:\n        if STAGE == "ci":\n            result = run_ci()\n        else:\n            actions = {"inventory": inventory, "fixtures": fixtures, "live": live_capture,\n                       "smoke": smoke, "connection": connection, "demo": demo,\n                       "efficiency": efficiency, "selection": selection, "stability": stability}\n            result = asyncio.run(actions[STAGE]())\n    except Exception as exc:\n        result = {"status": "FAIL", "error_type": type(exc).__name__, "error": str(exc),\n                  "traceback": traceback.format_exc()}\n    result.update(started_at=started, finished_at=utc(), action=STAGE,\n                  evaluated_sha=CFG["evaluated_sha"])\n    save(f"stages/{STAGE}.json", result)\n    print(json.dumps({k: v for k, v in result.items() if k not in ("trials", "runs", "checks", "cases", "traceback")},\n                     default=str, indent=2))\n    return 1 if result["status"] == "FAIL" else 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'

WORKER_PATH = RUN_ROOT / "release_worker.py"
WORKER_PATH.write_text(WORKER_SOURCE, encoding="utf-8")
(ARTIFACT_DIR / "release_worker.py").write_text(WORKER_SOURCE, encoding="utf-8")


In [ ]:
COHORT = {"topic": "CRISPR sickle cell", "fiscal_years": [2024], "funding_basis": "whole_award"}
QUERY_CRITERIA = {
    "advanced_text_search": {"operator": "and", "search_field": "all", "search_text": COHORT["topic"]},
    "fiscal_years": COHORT["fiscal_years"],
}
CI_SELECTORS = [
    "tests/test_counting.py",
    "tests/test_security.py",
    "tests/test_prompt_contracts.py",
    "tests/test_clinical_trials.py",
    "tests/test_version_contract.py",
    "tests/test_robustness.py::TestPartC_MockedUpstreamFailures",
]
with socket.socket() as reservation:
    reservation.bind(("127.0.0.1", 0))
    SERVER_PORT = reservation.getsockname()[1]
MCP_URL = f"http://127.0.0.1:{SERVER_PORT}/mcp"
CONFIG = {
    "run_id": RUN_ID, "repo_dir": str(REPO_DIR), "artifact_dir": str(ARTIFACT_DIR),
    "instrument_revision": "wrap-result-fix-2026-10-21",
    "target_scope": "local_pinned_http_server_not_deployed_app",
    "evaluated_sha": SOURCE_IDENTITY["evaluated_sha"], "release_version": RELEASE_VERSION,
    "expected_inventory": EXPECTED_INVENTORY,
    "mcp_url": MCP_URL, "cohort": COHORT, "query_criteria": QUERY_CRITERIA,
    "reporter_url": "https://api.reporter.nih.gov/v2/projects/search",
    "ci_selectors": CI_SELECTORS, "demo_model": DEMO_MODEL,
    "eval_models": EVAL_MODELS, "model_parameters": MODEL_PARAMETERS,
}
CONFIG_PATH = RUN_ROOT / "run_config.json"
def write_config():
    CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2))
    (ARTIFACT_DIR / "run_config.json").write_text(json.dumps(CONFIG, indent=2))
write_config()
STAGE_RESULTS = {}
EXECUTIONS = []
AGENT_ENV = {}

def run_stage(action, *, enabled=True, reason=None, extra_env=None, timeout=1800, quiet=False):
    stage_dir = ARTIFACT_DIR / "stages"
    stage_dir.mkdir(exist_ok=True)
    if not enabled:
        result = {"action": action, "status": "NOT_RUN", "reason": reason or "Disabled in configuration"}
        (stage_dir / f"{action}.json").write_text(json.dumps(result, indent=2))
    else:
        command = [sys.executable, "-S", str(WORKER_PATH), action, "--config", str(CONFIG_PATH)]
        attempt = 1 + sum(r["action"] == action for r in EXECUTIONS)
        attempt_dir = ARTIFACT_DIR / "attempts"
        attempt_dir.mkdir(exist_ok=True)
        log = attempt_dir / f"{action}_{attempt:02d}.log"
        record = {"action": action, "attempt": attempt, "command": command,
                  "log": str(log.relative_to(ARTIFACT_DIR)),
                  "started_at": datetime.now(timezone.utc).isoformat()}
        result_file = stage_dir / f"{action}.json"
        result_file.unlink(missing_ok=True)
        try:
            with log.open("w") as stream:
                proc = subprocess.Popen(command, cwd=REPO_DIR, env={**RUNTIME_ENV, **(extra_env or {})},
                                        stdout=stream, stderr=subprocess.STDOUT,
                                        start_new_session=(os.name == "posix"))
                try:
                    proc.wait(timeout=timeout)
                except subprocess.TimeoutExpired:
                    if os.name == "posix":
                        os.killpg(proc.pid, signal.SIGKILL)
                    else:
                        proc.kill()
                    proc.wait()
                    raise
            record["exit_code"] = proc.returncode
            result = json.loads(result_file.read_text()) if result_file.exists() else {
                "action": action, "status": "FAIL", "error": "Worker produced no stage report."}
            if proc.returncode and result.get("status") != "FAIL":
                result = {"action": action, "status": "FAIL", "worker_result": result,
                          "error": f"Worker exited with status {proc.returncode}"}
        except subprocess.TimeoutExpired:
            record["exit_code"] = None
            result = {"action": action, "status": "FAIL", "error": f"Stage exceeded {timeout} seconds"}
        except Exception as exc:
            record["exit_code"] = None
            result = {"action": action, "status": "FAIL", "error": f"{type(exc).__name__}: {exc}"}
        record["finished_at"] = datetime.now(timezone.utc).isoformat()
        EXECUTIONS.append(record)
        (attempt_dir / f"{action}_{attempt:02d}.json").write_text(json.dumps(result, indent=2))
        (stage_dir / f"{action}.json").write_text(json.dumps(result, indent=2))
    STAGE_RESULTS[action] = result
    if not quiet:
        print(f"{action}: {result['status']}")
        if result["status"] in {"FAIL", "BLOCKED"}:
            print(result.get("error", "See the stage JSON for individual failures."))
    return result


## 5. Start the local MCP server and verify its inventory

Only this notebook's server process is managed. Readiness requires a successful
MCP inventory exchange and matching /version response, not merely a PID or an
elapsed sleep. Counts are expected release contracts.

Startup probes can fail before the process begins listening. All attempts remain
in the archive; the displayed startup retries are not final inventory verdicts.
A final PASS requires the successful MCP and version checks. This does not probe
or stop an already deployed Databricks App.


In [ ]:
SERVER_LOG_HANDLE = (ARTIFACT_DIR / "server.log").open("w")
server_env = {**RUNTIME_ENV, "PORT": str(SERVER_PORT)}
SERVER_PROCESS = subprocess.Popen(
    [sys.executable, "-S", "-m", "nih_reporter_mcp.app"],
    cwd=REPO_DIR, env=server_env, stdout=SERVER_LOG_HANDLE, stderr=subprocess.STDOUT)
def stop_server():
    if SERVER_PROCESS.poll() is None:
        SERVER_PROCESS.terminate()
        try:
            SERVER_PROCESS.wait(timeout=10)
        except subprocess.TimeoutExpired:
            SERVER_PROCESS.kill()
            SERVER_PROCESS.wait(timeout=5)
    if not SERVER_LOG_HANDLE.closed:
        SERVER_LOG_HANDLE.close()
atexit.register(stop_server)

ready = False
for attempt in range(12):
    if SERVER_PROCESS.poll() is not None:
        break
    observed = run_stage("inventory", timeout=30, quiet=True)
    if observed["status"] == "PASS":
        ready = True
        break
    print(f"Startup probe {attempt + 1}: retrying; details retained in attempts/.")
    time.sleep(1)
if not ready:
    stop_server()
    print((ARTIFACT_DIR / "server.log").read_text()[-6000:])
    raise RuntimeError("Server readiness/inventory failed; verify against the running code.")
print("inventory: PASS (new local pinned server; deployed App not tested)")
print("Local MCP URL:", MCP_URL)
print(json.dumps(observed["observed"], indent=2))
print("Resource templates (separate from resources):", observed["resource_templates"])


## 6. Define the funding cohort

The demonstration uses **CRISPR sickle cell, FY2024**, with text-search
operator="and" and search_field="all". This is a reproducible keyword cohort,
not a program roster or official RCDC category.

Whole-award rules deduplicate application IDs first, exclude non-null
subproject IDs, and exclude null amounts without treating them as zero.
The ledger flags conflicting duplicate IDs and excluded children lacking a
same-core, same-FY parent. Award records and distinct core projects are different
counting units.


In [ ]:
print(json.dumps({"tool_arguments": COHORT, "raw_query_criteria": QUERY_CRITERIA}, indent=2))
print("Matching records, contributing records, and distinct award families are different quantities.")


## 7. Show and independently reconcile the frozen evidence

This checks hashes and independently recomputes both committed fixtures.
The small cohort's **historical comparison value** is $19,790,134; it is not
preloaded as a newly observed result. Raw snapshots and stored tool responses
may have different historical retrieval dates, both of which are preserved.

The two fixtures teach **opposite lessons** and must not be conflated:

- **U19AG068753 = double-counting proof.** Parent and children are both present;
  summing all records counts the same dollars twice, so whole_award is correct.
- **CRISPR sickle cell / P50HD096723 = orphan and completeness-limit proof.** The
  keyword matched one subproject (appl_id 10992993) but not its parent, so the
  parent is absent. Excluding that child removes **real topical investment, not
  a duplicate**: whole_award can **understate** a keyword cohort's investment.

The small cohort's all_records total printed below is a **run-time comparison,
not a cited release figure**. It equals whole_award plus that single orphan
subproject (listed in decision_ledger.csv). The difference is an orphan
exclusion, never double-counting removal.

The independent reducer does not call production aggregation. Production
comparison is separate. For the small cohort, the actual MCP tool is also
exercised in-process with a replay adapter and network requests blocked.
Raw projected fixtures do not independently establish organization/IC counts.


In [ ]:
fixture_result = run_stage("fixtures")
for fixture in fixture_result.get("fixtures", []):
    print("\nFixture:", fixture["fixture"])
    print("Historical source date:", fixture["historical_retrieved_at"])
    for basis, result in fixture["comparisons"].items():
        print(basis, result["total"], result["counts"])
        print("Orphan child IDs:", result["orphan_child_ids"])
    print("Scope:", fixture.get("scope", ""))


## 8. Fresh source capture → evidence bundle → independent reconciliation

Optional live section. Raw API pages, requests, timestamps and hashes are saved
first. Both bases then run through the **actual production MCP tool using the
same captured pages**, checked against an independent Decimal ledger.
This is **live capture followed by offline MCP replay**, not a direct deployed-app
test or an atomic NIH database snapshot.

The full contributing-ID ledger is saved separately. The tool's source_record_ids
lists up to 500 of N contributing IDs, capped for payload size;
records_aggregated gives the true count.


In [ ]:
live_result = run_stage("live", enabled=RUN_LIVE_CAPTURE,
                        reason="Fresh capture disabled; frozen replay remains available.")
if live_result["status"] == "PASS":
    print("Raw matching records:", live_result["total_matching"])
    for basis, result in live_result["comparisons"].items():
        print("\n", basis, result["total"])
        print("Decision counts:", result["counts"])
        print("Tool caveat:", result["tool_note"])
        print("Orphan children:", result["orphan_child_ids"])
        print("Conflicting IDs:", result["conflicting_ids"])
    print("Basis difference (% of whole_award):", live_result["basis_difference_percent"])
    print("\n".join(live_result["limits"]))


## 9. What the reconciled number does—and does not—establish

- A matching sum establishes arithmetic under stated rules for captured records.
  Completeness, missing values, conflicts, missing IDs and orphan components
  remain separate questions.
- Excluding an orphan component can omit associated award-family investment.
  Do not silently fetch and attribute an entire parent award to a topic.
- A basis difference is not proof that every excluded dollar was double-counted;
  the parent/component fixture supplies the specific overlap demonstration.
- No matches do not establish no funding or no relationship.
- Recipient-institution geography is not necessarily study location.
- Category-tagged aggregation is not automatically the published RCDC expenditure.
- Grant/output associations do not establish causal outcomes, fiscal-year
  attribution, or trial-level dollar attribution.
- Hashes establish artifact identity, not correctness of scientific attribution.


## 10. Reproduce the tagged model-free CI selection

This uses the six selections in the tagged workflow, including clinical trials
and the /version contract test. Expected: **155 passed + 2 xfailed**.
Counts come from pytest reports; xfail, skip, xpass, failure, error and not-run
are distinguished. JUnit XML and complete logs are saved.

The reporting plugin and disabled plugin autoload are explicit harness
differences. This is one Python runtime, not both CI matrix jobs. A count
mismatch is reported, never repaired during the run.


In [ ]:
ci_result = run_stage("ci", enabled=RUN_CI, reason="CI reproduction disabled.")
if ci_result["status"] != "NOT_RUN":
    print("Expected:", EXPECTED_CI)
    print("Observed:", ci_result.get("observed", {}).get("counts", {}))
    print("Exit code:", ci_result.get("exit_code"))
    print("JUnit:", ARTIFACT_DIR / "ci/junit.xml")


## 11. Notebook-defined live checks (not the 17-case standalone smoke suite)

**Two different suites, two different counts.** This section runs the
notebook's own live-check set; its count is generated by this run. The
repository's standalone `eval/smoke_tests.py` is a **separate 17-case suite**
(t1–t8, t10–t18) whose documented 17/17 result is unchanged; this notebook does
not execute it. The internal stage key `smoke` refers to the notebook-defined
set. The notebook's live-check count is neither a replacement for nor a
contradiction of the standalone 17/17, and it is not part of the CI figure.

These checks use upstream services and may fail because
of access restrictions, source changes or a real contract defect. Counts are
generated from executed checks.

The real clinical_trials_lookup tool is invoked; there is no stub-file import.
Mocked 403/429/timeout behavior remains in CI. No historical record count, WAF
response or active-award assumption is an unconditional live oracle.
Repeated-call consistency is an observation, not proof of cache hits or speedup.


In [ ]:
smoke_result = run_stage("smoke", enabled=RUN_LIVE_SMOKE, reason="Live checks disabled.")
print("Suite: notebook-defined live checks (distinct from the 17-case standalone eval/smoke_tests.py suite, not run here)")
for check in smoke_result.get("checks", []):
    print(check["status"], check["name"])
    if check["status"] != "PASS":
        print(" ", check.get("detail"))
print("Notebook-defined live-check counts:", smoke_result.get("counts", {}))


## 12. Optional Databricks endpoints — OFF by default

**Endpoint-dependent.** Subsequent model sections require authenticated
Databricks access and permission to invoke each named endpoint. They are
separate from model-free CI. Missing access is **NOT_RUN** or **BLOCKED**,
not a pass or an incorrect model answer.

Invocation cells are guarded by RUN_AGENT_EVALS. Endpoint names alone do not
establish tool-calling compatibility. Credentials are passed only to worker
processes and are not written to artifacts.


In [ ]:
AGENT_READY = False
if RUN_AGENT_EVALS:
    db_host = os.environ.get("DATABRICKS_HOST", "").rstrip("/")
    db_token = os.environ.get("DATABRICKS_TOKEN", "")
    if not (db_host and db_token):
        try:
            context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
            db_host = "https://" + spark.conf.get("spark.databricks.workspaceUrl")
            db_token = context.apiToken().getOrElse(None)
        except Exception:
            db_host, db_token = "", ""
    if db_host.startswith("https://") and db_token:
        AGENT_ENV = {"NIH_EVAL_DB_HOST": db_host, "NIH_EVAL_DB_TOKEN": db_token}
        preflight = run_stage("connection", extra_env=AGENT_ENV)
        AGENT_READY = preflight["status"] == "PASS"
    else:
        preflight = run_stage("connection", enabled=False,
                              reason="Databricks credentials unavailable; verify on the intended compute.")
else:
    preflight = run_stage("connection", enabled=False, reason="RUN_AGENT_EVALS=False")
print("Model preflight passed:", AGENT_READY)


## 13. Optional explicit model selection

Discovery permissions may differ from invocation permissions. On Databricks,
the widget lists configured endpoints that passed this run's tool-calling probe.
Outside Databricks the explicit DEMO_MODEL configuration is used.


In [ ]:
if AGENT_READY:
    passed_models = [r["model"] for r in preflight["endpoints"] if r["status"] == "PASS"]
    try:
        dbutils.widgets.dropdown("nih_eval_model", DEMO_MODEL, passed_models, "Verified probe endpoint")
        selected_model = dbutils.widgets.get("nih_eval_model")
    except NameError:
        selected_model = DEMO_MODEL
    except Exception as exc:
        print("Widget unavailable; using configured model:", type(exc).__name__)
        selected_model = DEMO_MODEL
    if selected_model not in passed_models:
        raise RuntimeError("Selected model did not pass this run's tool-calling probe.")
    CONFIG["demo_model"] = selected_model
    write_config()
    print("Selected model:", selected_model)
else:
    print("NOT_RUN: optional model section disabled or unavailable.")


## 14. Optional agent demonstrations

**Endpoint-dependent.** Three questions demonstrate evidence export,
category-versus-text comparison and trial linkage boundaries. Two successful
comparison query paths are checked, not assumed to be forced by the prompt.
Full tool outputs and the exact model-visible projections are saved.

The projection preserves nested scalar caveats, always returns valid JSON and
records omitted arrays. Its character budget is soft when caveats require more
space. Generated answers remain candidates for human review; truth is not
automatically scored.


In [ ]:
demo_result = run_stage("demo", enabled=AGENT_READY,
                        reason="Model preflight disabled or unavailable.", extra_env=AGENT_ENV)
for item in demo_result.get("runs", []):
    print("\nQUESTION:", item["question"])
    print("Run status:", item["status"])
    print(item.get("answer") or item.get("error"))
    if "comparison_queries_observed" in item:
        print("Both comparison query paths observed:", item["comparison_queries_observed"])


## 15. Optional tool-call efficiency

**Endpoint-dependent.** Three fresh conversations measure tool-call count,
not model-round count. Eligibility requires an answer, a successful expected
tool call and no failed tool calls. Failed/ineligible runs remain visible.
Efficiency does not establish answer correctness.


In [ ]:
efficiency_result = run_stage("efficiency", enabled=AGENT_READY,
                              reason="Model preflight disabled or unavailable.", extra_env=AGENT_ENV)
print(json.dumps({k: v for k, v in efficiency_result.items()
                  if k not in {"runs", "traceback"}}, indent=2))


## 16. Optional first-tool selection and argument evaluation

**Endpoint-dependent.** The original eight questions form a named regression
slice across three models. Revised instructions and schema handling change the
protocol; do not compare percentages with earlier runs as if it were identical.

Selection and argument scoring use the **same captured response**. Expected
answers are not sent to target models. Endpoint errors, no-tool responses,
wrong selections, malformed JSON and schema violations are distinguished.
Full JSON Schema validation handles nested constraints, unions and bounds.

Runtime admissibility is not inferred from schema validity. Known source-inspected
cross-field constraints are checked, but this is not a complete runtime-validity
benchmark. A project lookup with both identifiers null is flagged.
Final answers and the entire 14-tool surface are not scored. Q8 tests tool
selection, not whether a last-week date filter can be enforced.


In [ ]:
selection_result = run_stage("selection", enabled=AGENT_READY,
                             reason="Model preflight disabled or unavailable.", extra_env=AGENT_ENV)
if selection_result["status"] != "NOT_RUN":
    fields = [
        "planned_trials", "recorded_trials", "endpoint_errors", "evaluation_errors", "selection_correct",
        "selection_denominator_all_planned", "selection_denominator_completed_responses",
        "schema_valid_calls", "schema_validity_denominator_generated_first_calls",
        "known_runtime_rejections", "runtime_admissibility",
        "unanimous_correct_cases", "unanimous_denominator",
    ]
    print(json.dumps({k: selection_result.get(k) for k in fields}, indent=2))


## 17. Optional observed selection consistency

**Endpoint-dependent.** Three questions are repeated five times. Repeated errors
or no-tool responses never count as consistent successful selection.
Consistency and correctness are separate. This sample does not prove general
determinism. Actual settings and returned model identifiers are retained.


In [ ]:
stability_result = run_stage("stability", enabled=AGENT_READY,
                             reason="Model preflight disabled or unavailable.", extra_env=AGENT_ENV)
print(json.dumps({k: v for k, v in stability_result.items()
                  if k not in {"cases", "traceback"}}, indent=2))


## 18. Observability and evidence status

Tool-call success means invocation, not scientific validity. An invalid-input
check may intentionally generate an error event. Logs are scoped by stage/run
and never cleared to hide earlier attempts.

- **Verified in this run:** a completed check with saved supporting artifacts,
  limited to its stated scope.
- **Repository-reported:** a repository/release assertion not reproduced here.
- **Team-reported:** an assertion outside inspectable repository/run evidence.
- **NOT_RUN / BLOCKED / FAIL:** execution states, never counted as passes.


In [ ]:
from collections import Counter
tool_log = ARTIFACT_DIR / "tool_calls.jsonl"
events = [json.loads(line) for line in tool_log.read_text().splitlines()] if tool_log.exists() else []
for stage in sorted({e["stage"] for e in events}):
    group = [e for e in events if e["stage"] == stage]
    print(stage, dict(Counter(e["status"] for e in group)), "calls:", len(group))
print("\nSTAGES")
for name, result in STAGE_RESULTS.items():
    print(f"{name:14} {result['status']}")
print("\nExpected release CI baseline:", EXPECTED_CI)
print("Observed local CI:", STAGE_RESULTS.get("ci", {}).get("observed", {}).get("counts", "NOT_RUN"))


## 19. Package this execution, then enforce the result

The manifest records source identity, environment, actual results, commands,
capture/replay distinctions, cache state and hashes. It never promotes inherited
outputs to new evidence. The archive includes the worker and unsuccessful trials.
The worker hash identifies this revised instrument, separately from the
evaluated server SHA.


In [ ]:
stop_server()
tracked_changes = checked_command(["git", "status", "--porcelain", "--untracked-files=no"], cwd=REPO_DIR)
STAGE_RESULTS["source_integrity"] = (
    {"status": "FAIL", "changes": tracked_changes} if tracked_changes else {"status": "PASS"})
artifact_hashes = {
    str(p.relative_to(ARTIFACT_DIR)): hashlib.sha256(p.read_bytes()).hexdigest()
    for p in sorted(ARTIFACT_DIR.rglob("*")) if p.is_file()
    and p.name not in {"manifest.json", "SHA256SUMS"}
}
manifest = {
    "run_id": RUN_ID, "finished_at": datetime.now(timezone.utc).isoformat(),
    "source_identity": SOURCE_IDENTITY, "environment": ENVIRONMENT,
    "instrument": {"worker_sha256": hashlib.sha256(WORKER_SOURCE.encode()).hexdigest(),
                   "revision": CONFIG["instrument_revision"],
                   "relationship_to_release": "Revised instrument; server source unchanged"},
    "evaluation_target": {"scope": CONFIG["target_scope"], "mcp_url": MCP_URL,
                          "deployed_app_tested": False},
    "flags": {"ci": RUN_CI, "live_capture": RUN_LIVE_CAPTURE,
              "live_smoke": RUN_LIVE_SMOKE, "agent_evaluations": RUN_AGENT_EVALS},
    "cache_state": {
        "http_server": "Fresh process; cache enabled, then warm during this run",
        "frozen_replay": "HTTP blocked; explicit recorded-response adapter",
        "live_capture": "Independent HTTP; each request/response recorded",
    },
    "stage_results": {k: {"status": v["status"], "result_path": f"stages/{k}.json"}
                      for k, v in STAGE_RESULTS.items() if k != "source_integrity"},
    "source_integrity": STAGE_RESULTS["source_integrity"],
    "executions": EXECUTIONS, "artifact_sha256": artifact_hashes,
    "suite_scopes": {
        "smoke": ("Notebook-defined live checks; count generated by this run. "
                  "Distinct from the 17-case standalone eval/smoke_tests.py suite, "
                  "which this notebook does not execute."),
        "ci": "Tagged model-free CI selection (ci.yml selectors).",
    },
    "limits": [
        "No deployed-app Ask acceptance is claimed.",
        "The standalone 17-case smoke suite is not executed by this notebook.",
        "Model answers are not independently adjudicated.",
        "Fresh dependency resolution is not a committed dependency lock.",
        "Live capture/replay differs from direct live HTTP tool invocation.",
    ],
}
(ARTIFACT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
hash_lines = []
for path in sorted(ARTIFACT_DIR.rglob("*")):
    if path.is_file() and path.name != "SHA256SUMS":
        hash_lines.append(f"{hashlib.sha256(path.read_bytes()).hexdigest()}  {path.relative_to(ARTIFACT_DIR)}")
(ARTIFACT_DIR / "SHA256SUMS").write_text("\n".join(hash_lines) + "\n")
ARCHIVE_PATH = Path(shutil.make_archive(str(ARTIFACT_DIR.parent / (RUN_ID + "-evidence")),
                                      "zip", root_dir=ARTIFACT_DIR))
print("Evidence archive:", ARCHIVE_PATH)
print("Manifest:", ARTIFACT_DIR / "manifest.json")
try:
    from IPython.display import FileLink, display
    display(FileLink(str(ARCHIVE_PATH)))
except ImportError:
    pass
required = ["inventory", "fixtures", "source_integrity"]
required += ["ci"] if RUN_CI else []
required += ["live"] if RUN_LIVE_CAPTURE else []
required += ["smoke"] if RUN_LIVE_SMOKE else []
failed = [name for name in required if STAGE_RESULTS.get(name, {}).get("status") != "PASS"]
if failed:
    raise AssertionError(f"Required checks failed: {failed}. Evidence preserved in {ARCHIVE_PATH}")
print("Required core checks passed in this execution. Optional agent status is reported separately.")


## What to show a judge

1. Resolved source identity and observed 14/9/3 inventory.
2. One explicitly defined funding cohort and its evidence.
3. Independent reconciliation and visible limits.
4. Actual CI and notebook-defined live-check results with their different scopes.
   The notebook's live-check count is not the standalone smoke suite's 17/17.
5. Optional agent behavior, endpoint dependencies and unscored answer correctness.

Until executed, this file contains **no new evaluation results**. An expected
baseline is not an observed metric.

Release references:
[run manifest](https://github.com/conroyriodss/nih-reporter-mcp/blob/v0.14.6/docs/demonstration_fixture/RUN_MANIFEST.md),
[release](https://github.com/conroyriodss/nih-reporter-mcp/releases/tag/v0.14.6),
[CI workflow](https://github.com/conroyriodss/nih-reporter-mcp/blob/v0.14.6/.github/workflows/ci.yml).
